In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

In [23]:
data = pd.read_csv("../data/clean_traffic.csv")
df = pd.DataFrame(data)
df

,DateTime,Junction,Vehicles,ID,Hour,Month,Day,Weekday
0,2015-11-01 00:00:00,1,15,20151101001,0,11,1,6
1,2015-11-01 01:00:00,1,13,20151101011,1,11,1,6
2,2015-11-01 02:00:00,1,10,20151101021,2,11,1,6
3,2015-11-01 03:00:00,1,7,20151101031,3,11,1,6
4,2015-11-01 04:00:00,1,9,20151101041,4,11,1,6
...,...,...,...,...,...,...,...,...
48115,2017-06-30 19:00:00,4,11,20170630194,19,6,30,4
48116,2017-06-30 20:00:00,4,30,20170630204,20,6,30,4
48117,2017-06-30 21:00:00,4,16,20170630214,21,6,30,4
48118,2017-06-30 22:00:00,4,22,20170630224,22,6,30,4


In [24]:
df["DateTime"] = pd.to_datetime(df["DateTime"])
df['Hour'] = df['DateTime'].dt.hour
df['Day'] = df['DateTime'].dt.day
df['Month'] = df['DateTime'].dt.month
df['Weekday'] = df['DateTime'].dt.weekday

In [25]:
df = df.sort_values(by=['Junction', 'DateTime'])

In [26]:
df['Lag_1'] = df.groupby('Junction')['Vehicles'].shift(1)
df['Lag_2'] = df.groupby('Junction')['Vehicles'].shift(2)
df['Lag_3'] = df.groupby('Junction')['Vehicles'].shift(3)

In [27]:
df['Rolling_Mean_3'] = (
    df.groupby('Junction')['Vehicles']
    .rolling(window=3)
    .mean()
    .reset_index(0, drop=True)
)

In [28]:
df.columns

Index(['DateTime', 'Junction', 'Vehicles', 'ID', 'Hour', 'Month', 'Day',
       'Weekday', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3'],
      dtype='str')

In [29]:
features = [
    'Junction',
    'Hour',
    'Day',
    'Month',
    'Weekday',
    'Lag_1',
    'Lag_2',
    'Lag_3',
    'Rolling_Mean_3'
]
X = df[features]
y = df['Vehicles']

In [30]:
split_index = int(len(X) * 0.8)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

In [31]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [32]:
predictions = model.predict(X_test)

In [33]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

In [34]:
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.2f}")

Mean Absolute Error: 0.43
Mean Squared Error: 3.50
R² Score: 0.96


In [35]:
results = pd.DataFrame({
    "Actual" : y_test,
    "Predicted" : predictions
})
results.head()

,Actual,Predicted
38496,17,16.89
38497,12,11.80
38498,10,9.73
38499,7,6.92
38500,4,4.39


In [36]:
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
})

feature_importance.sort_values(
    by='Importance',
    ascending=False
)

,Feature,Importance
8,Rolling_Mean_3,0.973615
6,Lag_2,0.013476
1,Hour,0.006116
5,Lag_1,0.004657
7,Lag_3,0.000870
2,Day,0.000355
0,Junction,0.000353
3,Month,0.000315
4,Weekday,0.000243


In [37]:
joblib.dump(
    model, "../models/traffic_model.pkl"
)

print("Model saved successfully 🚀")

Model saved successfully 🚀
